In [ ]:
"""
Strategy 4 — VWAP Reclaim
═════════════════════════
Popular 2023–2025. VWAP = institutional average price.

CONCEPT:
  When price dips below VWAP then reclaims it with volume,
  institutions are re-entering → bullish bias.

SETUP:
  1. Stock was trading above VWAP earlier today
  2. Dipped below VWAP (at least 2 bars below)
  3. Reclaims VWAP with a strong candle (close > VWAP, close > open)
  4. Volume on reclaim bar > 1.5x average

ENTRY CONDITIONS (all required):
  • Price crossed back above VWAP on current bar
  • Prior 2+ bars were below VWAP (confirmed dip, not noise)
  • Reclaim candle is green (close > open)
  • Volume > 1.5x 20-bar average
  • EMA 9 > EMA 21 (broader trend still bullish)

EXIT:
  • Stop = low of the dip (below VWAP)
  • Target = day's high or 2R
  • Hard stop: -5%
  • Best in: mid-day reversals, trending markets
"""

import asyncio
import sys
import random
import logging
import warnings
from datetime import datetime, timezone, timedelta

import nest_asyncio
nest_asyncio.apply()

import numpy as np
import pandas as pd
from pymongo import MongoClient
from bson import ObjectId
from ib_insync import IB, Stock, MarketOrder, LimitOrder, util

warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

IBKR_HOST      = "127.0.0.1"
IBKR_PORT      = 7497
IBKR_CLIENT_ID = random.randint(1, 1000)
ORDER_SIZE     = 20

# VWAP reclaim parameters
MIN_BARS_BELOW_VWAP  = 2     # need at least 2 bars below VWAP before reclaim
RECLAIM_VOL_MULT     = 1.5   # reclaim bar needs 1.5x avg volume
HARD_STOP_PCT        = 0.05
TARGET_R_MULTIPLE    = 2.0

MIN_PRICE = 2.0
MAX_PRICE = 30.0

SCAN_INTERVAL    = 300
RTH_START_HOUR   = 9
RTH_START_MINUTE = 30
RTH_END_HOUR     = 16
LIMIT_SLIPPAGE   = 0.01


# ══════════════════════════════════════════════════════════════════════════════
# LOGGING + INFRA
# ══════════════════════════════════════════════════════════════════════════════

class _Utf8SafeStreamHandler(logging.StreamHandler):
    def emit(self, record):
        try: super().emit(record)
        except UnicodeEncodeError:
            record.msg = str(record.msg).encode("ascii", errors="replace").decode("ascii")
            record.args = ()
            try: super().emit(record)
            except Exception: self.handleError(record)

def _make_logger(name):
    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    sh = _Utf8SafeStreamHandler(sys.stdout); sh.setFormatter(fmt)
    fh = logging.FileHandler(f"{name}.log", encoding="utf-8"); fh.setFormatter(fmt)
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG); logger.handlers.clear()
    logger.addHandler(sh); logger.addHandler(fh); logger.propagate = False
    return logger

log = _make_logger("strat4_vwap_reclaim")

mongo  = MongoClient("mongodb://localhost:27017/")
db     = mongo["breakout_db"]
trades = db["trades_v2"]

ib = IB()

def ibkr_connect():
    util.startLoop()
    ib.connect(IBKR_HOST, IBKR_PORT, clientId=IBKR_CLIENT_ID)
    log.info(f"IBKR connected | accounts: {ib.wrapper.accounts}")

def _get_eastern_now():
    utc = datetime.now(timezone.utc)
    return utc + (timedelta(hours=-4) if 3 <= utc.month <= 10 else timedelta(hours=-5))

def is_rth():
    et = _get_eastern_now()
    o = et.replace(hour=RTH_START_HOUR, minute=RTH_START_MINUTE, second=0, microsecond=0)
    c = et.replace(hour=RTH_END_HOUR, minute=0, second=0, microsecond=0)
    return o <= et < c

def create_buy_order(qty, lp=None):
    if is_rth(): return MarketOrder("BUY", qty)
    p = round(lp * (1 + LIMIT_SLIPPAGE), 2)
    o = LimitOrder("BUY", qty, p); o.outsideRth = True; o.tif = "DAY"; return o

def create_sell_order(qty, lp=None):
    if is_rth(): return MarketOrder("SELL", qty)
    p = round(lp * (1 - LIMIT_SLIPPAGE), 2)
    o = LimitOrder("SELL", qty, p); o.outsideRth = True; o.tif = "DAY"; return o

def has_open_position(s): return any(p.contract.symbol == s and p.position != 0 for p in ib.positions())
def has_pending_order(s): return any(t.contract.symbol == s for t in ib.openTrades())

def get_ask_price(c):
    t = ib.reqMktData(c, "106", False, False); ib.sleep(3); ib.cancelMktData(c)
    for _, v in [("ask", t.ask), ("last", t.last), ("close", t.close)]:
        if v and float(v) > 0: return float(v)
    return None

def get_bid_price(c):
    t = ib.reqMktData(c, "106", False, False); ib.sleep(3); ib.cancelMktData(c)
    for _, v in [("bid", t.bid), ("last", t.last), ("close", t.close)]:
        if v and float(v) > 0: return float(v)
    return None


# ══════════════════════════════════════════════════════════════════════════════
# VWAP RECLAIM DETECTION
# ══════════════════════════════════════════════════════════════════════════════

def add_indicators(df):
    df = df.copy()

    # Session VWAP
    df["date_only"] = pd.to_datetime(df["date"]).dt.date
    today = df["date_only"].max()
    mask = df["date_only"] == today
    tdf = df.loc[mask].copy()
    if len(tdf) > 0 and tdf["volume"].sum() > 0:
        tdf["vwap"] = (tdf["close"] * tdf["volume"]).cumsum() / tdf["volume"].cumsum()
        df.loc[mask, "vwap"] = tdf["vwap"].values
    df["vwap"] = df["vwap"].ffill()

    # EMAs
    df["ema_9"]  = df["close"].ewm(span=9, adjust=False).mean()
    df["ema_21"] = df["close"].ewm(span=21, adjust=False).mean()

    # Volume average
    df["vol_avg"] = df["volume"].rolling(20).mean()

    # Track if bar is above/below VWAP
    df["above_vwap"] = df["close"] > df["vwap"]

    return df


def detect_vwap_reclaim(df: pd.DataFrame, symbol: str) -> dict | None:
    """
    Detect a VWAP reclaim pattern:
    1. Recent bars were below VWAP
    2. Current bar crosses back above VWAP
    3. Reclaim has volume + is a green candle
    4. Broader trend (EMA 9 > 21) still bullish
    """
    if len(df) < 10:
        return None

    # Get today's bars only
    df_copy = df.copy()
    df_copy["date_only"] = pd.to_datetime(df_copy["date"]).dt.date
    today = df_copy["date_only"].max()
    today_bars = df_copy[df_copy["date_only"] == today]

    if len(today_bars) < 5:
        return None

    row = df.iloc[-1]
    price   = float(row["close"])
    vwap    = float(row.get("vwap", 0))
    vol     = float(row["volume"])
    vol_avg = float(row.get("vol_avg", 1))

    if vwap <= 0:
        return None

    # Current bar must be ABOVE VWAP
    if price <= vwap:
        return None

    # Current bar must be green (close > open)
    if float(row["close"]) <= float(row["open"]):
        return None

    # Check that N prior bars were BELOW VWAP (the dip)
    lookback = df.iloc[-MIN_BARS_BELOW_VWAP - 1:-1]
    bars_below = (lookback["close"] < lookback["vwap"]).sum()
    if bars_below < MIN_BARS_BELOW_VWAP:
        return None

    # Volume confirmation
    vol_ratio = vol / vol_avg if vol_avg > 0 else 0
    if vol_ratio < RECLAIM_VOL_MULT:
        return None

    # EMA trend check (broader trend still bullish)
    ema_9  = float(row.get("ema_9", 0))
    ema_21 = float(row.get("ema_21", 0))
    if ema_9 <= ema_21:
        return None

    # Calculate stop and target
    dip_low = float(today_bars.loc[today_bars["close"] < today_bars["vwap"], "low"].min())
    if pd.isna(dip_low):
        dip_low = price * (1 - HARD_STOP_PCT)

    stop_price = dip_low * 0.995
    risk       = price - stop_price
    today_high = float(today_bars["high"].max())
    target     = max(today_high, price + risk * TARGET_R_MULTIPLE)

    return {
        "symbol":     symbol,
        "price":      price,
        "vwap":       round(vwap, 4),
        "bars_below": int(bars_below),
        "vol_ratio":  round(vol_ratio, 2),
        "ema_9":      round(ema_9, 4),
        "ema_21":     round(ema_21, 4),
        "dip_low":    round(dip_low, 4),
        "stop":       round(stop_price, 4),
        "target":     round(target, 4),
    }


# ══════════════════════════════════════════════════════════════════════════════
# TRADE DOCUMENT + SCANNER
# ══════════════════════════════════════════════════════════════════════════════

def create_trade_doc(sym, entry, qty, signal):
    return {
        "instrument":      sym,
        "strategy":        "vwap_reclaim",
        "direction":       "long",
        "entry_price":     entry,
        "highest_price":   entry,
        "quantity":        qty,
        "entry_timestamp": datetime.now(),
        "vwap_signal":     signal,
        "stop_price":      signal["stop"],
        "target_price":    signal["target"],
        "order_id":        str(ObjectId()),
        "status":          "open",
        "phase":           "initial",
        "created_at":      datetime.now(),
        "updated_at":      datetime.now(),
    }

def update_trade(tid, updates):
    updates["updated_at"] = datetime.now()
    trades.update_one({"_id": tid}, {"$set": updates})

def scan_active_stocks():
    from ib_insync import ScannerSubscription
    sub = ScannerSubscription(
        instrument="STK", locationCode="STK.US.MAJOR",
        scanCode="MOST_ACTIVE", abovePrice=MIN_PRICE, belowPrice=MAX_PRICE,
        aboveVolume=100000, numberOfRows=20,
    )
    try:
        results = ib.reqScannerData(sub); ib.sleep(2)
        return [r.contractDetails.contract.symbol for r in results]
    except Exception as e:
        log.warning(f"Scanner error: {e}"); return []


# ══════════════════════════════════════════════════════════════════════════════
# MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

async def trading_loop():
    log.info("VWAP Reclaim strategy started.")
    await asyncio.sleep(5)

    while True:
        et = _get_eastern_now()

        # Best for mid-day reversals: 10:30–15:00 ET
        if not is_rth() or et.hour < 10:
            log.info(f"Waiting for mid-day window (ET: {et.strftime('%H:%M')})")
            await asyncio.sleep(60)
            continue

        symbols = scan_active_stocks()
        log.info(f"Scanning {len(symbols)} active stocks for VWAP reclaims")

        for sym in symbols:
            contract = Stock(sym, "SMART", "USD")
            try: ib.qualifyContracts(contract)
            except: continue

            try:
                bars = ib.reqHistoricalData(
                    contract, endDateTime="", durationStr="2 D",
                    barSizeSetting="5 mins", whatToShow="TRADES",
                    useRTH=False, formatDate=1,
                )
            except: continue
            if not bars: continue

            df = util.df(bars)
            df.columns = [c.lower() for c in df.columns]
            df = add_indicators(df)

            signal = detect_vwap_reclaim(df, sym)
            if not signal:
                await asyncio.sleep(0.3)
                continue

            log.info(
                f"{sym}: VWAP RECLAIM | ${signal['price']:.2f} > VWAP ${signal['vwap']:.2f} | "
                f"bars_below={signal['bars_below']} | vol={signal['vol_ratio']}x | "
                f"stop=${signal['stop']:.2f} target=${signal['target']:.2f}"
            )

            if has_open_position(sym) or has_pending_order(sym): continue
            if trades.find_one({"instrument": sym, "status": "open"}): continue

            ask = get_ask_price(contract)
            if not ask: continue

            order = create_buy_order(ORDER_SIZE, lp=ask)
            ib.placeOrder(contract, order)
            trades.insert_one(create_trade_doc(sym, ask, ORDER_SIZE, signal))
            log.info(f"BUY | {sym} | ${ask:.2f} | VWAP reclaim")
            await asyncio.sleep(0.5)

        # ── Manage open trades ────────────────────────────────────────────
        for trade in trades.find({"strategy": "vwap_reclaim", "status": "open"}):
            sym   = trade["instrument"]
            entry = trade["entry_price"]
            qty   = trade["quantity"]
            stop  = trade.get("stop_price", entry * (1 - HARD_STOP_PCT))
            tgt   = trade.get("target_price", entry * 1.10)

            contract = Stock(sym, "SMART", "USD")
            try: ib.qualifyContracts(contract)
            except: continue

            bid = get_bid_price(contract)
            if not bid: continue

            pnl = (bid - entry) / entry
            update_trade(trade["_id"], {"highest_price": max(trade.get("highest_price", entry), bid)})

            reason = None
            if bid <= stop: reason = "vwap_stop"
            elif bid >= tgt: reason = f"vwap_target_{pnl*100:.1f}pct"
            elif pnl <= -HARD_STOP_PCT: reason = "hard_stop"

            if reason:
                ib.placeOrder(contract, create_sell_order(qty, lp=bid))
                update_trade(trade["_id"], {
                    "exit_price": bid, "exit_timestamp": datetime.now(),
                    "reason": reason, "realized_pnl": (bid - entry) * qty, "status": "closed",
                })
                log.info(f"SELL | {sym} | ${bid:.2f} | pnl={pnl*100:+.2f}% | {reason}")

        log.info(f"Scan complete. Next in {SCAN_INTERVAL//60} min.")
        await asyncio.sleep(SCAN_INTERVAL)


async def main():
    log.info("=== Strategy 4: VWAP Reclaim ===")
    ibkr_connect()
    await trading_loop()

if __name__ == "__main__":
    try: asyncio.run(main())
    except KeyboardInterrupt: log.info("Stopped.")
    finally: ib.disconnect()


2026-03-31 13:50:26,635 [INFO] === Strategy 4: VWAP Reclaim ===
2026-03-31 13:50:27,110 [INFO] IBKR connected | accounts: ['DUD087866']
2026-03-31 13:50:27,111 [INFO] VWAP Reclaim strategy started.


Error 162, reqId 3: Historical Market Data Service error message:API scanner subscription cancelled: 3


2026-03-31 13:50:34,666 [INFO] Scanning 20 active stocks for VWAP reclaims
2026-03-31 13:50:46,652 [INFO] Scan complete. Next in 5 min.


Error 162, reqId 44: Historical Market Data Service error message:API scanner subscription cancelled: 44


2026-03-31 13:55:49,154 [INFO] Scanning 20 active stocks for VWAP reclaims
2026-03-31 13:55:59,335 [INFO] Scan complete. Next in 5 min.


...

2026-03-31 15:45:23,437 [INFO] Scanning 20 active stocks for VWAP reclaims
2026-03-31 15:45:33,814 [INFO] Scan complete. Next in 5 min.


Error 162, reqId 946: Historical Market Data Service error message:API scanner subscription cancelled: 946


2026-03-31 15:50:36,199 [INFO] Scanning 20 active stocks for VWAP reclaims
2026-03-31 15:50:46,693 [INFO] Scan complete. Next in 5 min.


Error 162, reqId 987: Historical Market Data Service error message:API scanner subscription cancelled: 987


2026-03-31 15:55:49,116 [INFO] Scanning 20 active stocks for VWAP reclaims
2026-03-31 15:55:59,897 [INFO] Scan complete. Next in 5 min.
